In [1]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os

In [2]:
def get_dests(raw_data_paths, competition, seasons):
    """
    Get the destination paths for each season.
    """
    return [f"{raw_data_paths}/{competition}/{season}" for season in seasons]

In [3]:
def scrape_competitions_for_seasons(all_seasons, raw_data_path, competitions, urls):
    for competition in competitions:
        for season, dest in zip(all_seasons, get_dests(raw_data_path, competition, all_seasons)):
            url = urls[competition][season]
            print(f"Scraping {url} to {dest}")
            # Run the scraper
            #scrape_season_match_data(url, dest)

In [4]:
def get_processed_path(processed_data_path, competition):
    """
    Get the processed data path for a given competition.
    """
    return f"{processed_data_path}/{competition}"

In [5]:
def check_if_target_df_exists(processed_path, all_matches_df):
    """
    Check if the target DataFrame already exists and matches the expected length.
    """
    target_path = f"{processed_path}/all_target_df.csv"
    if os.path.exists(target_path):
        df = pd.read_csv(target_path)
        return len(df) == len(all_matches_df)
    return False

def check_if_data_df_exists(processed_path, all_matches_df):
    """
    Check if the data DataFrame already exists and matches the expected length.
    """
    data_path = f"{processed_path}/all_data_df.csv"
    if os.path.exists(data_path):
        df = pd.read_csv(data_path)
        return len(df) == len(all_matches_df)
    return False

In [6]:
def process_raw_match_data(all_competitions, all_seasons, raw_data_path, processed_data_path, target_columns, verbose=True):
    for competition in all_competitions:
        for season in all_seasons:
            season_path = f'{raw_data_path}/{competition}/{season}/'
            processed_path = f'{get_processed_path(processed_data_path, competition)}/{season}/'
            if not os.path.exists(processed_path):
                os.makedirs(processed_path)
            all_matches = pd.read_csv(f'{season_path}/all_matches.csv').iloc[:, 1].tolist()
            # Check if target df exists and matches length
            if check_if_target_df_exists(processed_path, all_matches):
                if verbose:
                    print(f"Target DataFrame for {competition} {season} already exists and matches. Skipping.")
            else:
                if verbose:
                    print(f"Processing target DataFrame for {competition} {season}.")
                target_df = []
                for match_name in all_matches:
                    home_df = pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
                    away_df = pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
                    match_df = pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
                    target_df.append(process_match_target_var(home_df, away_df, match_df, match_name))
                target_df = pd.concat(target_df, axis=0)
                target_df.to_csv(f'{processed_path}/all_target_df.csv', index=False)

        for season in all_seasons:
            season_path = f'{raw_data_path}/{competition}/{season}/'
            processed_path = f'{get_processed_path(processed_data_path, competition)}/{season}/'
            all_matches = pd.read_csv(f'{season_path}/all_matches.csv').iloc[:, 1].tolist()
            # Check if data df exists and matches length
            if check_if_data_df_exists(processed_path, all_matches):
                if verbose:
                    print(f"Data DataFrame for {competition} {season} already exists and matches. Skipping.")
            else:
                if verbose:
                    print(f"Processing data DataFrame for {competition} {season}.")
                data_df = []
                for match_name in all_matches:
                    home_df = pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
                    away_df = pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
                    match_df = pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
                    data_df.append(process_match_other_var(home_df, away_df, match_df, match_name, target_columns))
                data_df = pd.concat(data_df, axis=0)
                data_df.to_csv(f'{processed_path}/all_data_df.csv', index=False)

In [7]:
f'{RAW_DATA_PATH}/{COMPETITIONS[0]}/{ALL_SEASONS[0]}/'

'/Users/tianqihuang/Documents/GitHub/betbot//data/raw//premier_league/2017-18/'

In [8]:
COMPETITIONS

('premier_league',)

In [9]:
process_raw_match_data(COMPETITIONS, ALL_SEASONS, RAW_DATA_PATH, PROCESSED_DATA_PATH, TARGET_COLUMNS)

Target DataFrame for premier_league 2017-18 already exists and matches. Skipping.
Target DataFrame for premier_league 2018-19 already exists and matches. Skipping.
Target DataFrame for premier_league 2019-20 already exists and matches. Skipping.
Target DataFrame for premier_league 2020-21 already exists and matches. Skipping.
Target DataFrame for premier_league 2021-22 already exists and matches. Skipping.
Target DataFrame for premier_league 2022-23 already exists and matches. Skipping.
Target DataFrame for premier_league 2023-24 already exists and matches. Skipping.
Target DataFrame for premier_league 2024-25 already exists and matches. Skipping.
Data DataFrame for premier_league 2017-18 already exists and matches. Skipping.
Data DataFrame for premier_league 2018-19 already exists and matches. Skipping.
Data DataFrame for premier_league 2019-20 already exists and matches. Skipping.
Data DataFrame for premier_league 2020-21 already exists and matches. Skipping.
Data DataFrame for premi

In [10]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"

In [11]:
encoder = TeamEncoder.load(team_encoder_path)

In [12]:
competition = COMPETITIONS[0]

In [13]:
seasons=sorted(ALL_SEASONS)
data_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons]
target_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]

In [27]:
test_df=pd.DataFrame({
    'home': ['Chelsea'],
    'away': ['Bournemouth'],
    'date': ['2024-05-19']
    })

In [28]:
CURRENT_SEASON

'2024-25'

In [29]:
# Prepare season_dfs as a dict for transform_spot
season_dfs_dict = {season: df for season, df in zip(seasons, data_dfs)}
# Run transform_spot for the current season and test_df
spot_features = encoder.transform_spot(season_dfs_dict, CURRENT_SEASON, test_df)
spot_features

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [24]:
already_feature_df= pd.read_csv(f"../../data/features/premier_league/all_combined_features_2017-24.csv")

In [30]:
already_feature_df['date'].max()

'2024-05-19'

In [26]:
already_feature_df[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth')]

,home,away,date,encoded_home_Arsenal,encoded_home_Bournemouth,encoded_home_Brighton,encoded_home_Burnley,encoded_home_Chelsea,encoded_home_Crystal Palace,encoded_home_Everton,...,away_lag_5_home_goals,away_lag_5_away_goals,away_lag_5_home_corners,away_lag_5_away_corners,away_lag_5_home_cards,away_lag_5_away_cards,away_lag_5_home_shots,away_lag_5_away_shots,away_lag_5_home_sots,away_lag_5_away_sots
346,Chelsea,Bournemouth,2018-09-01,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
426,Chelsea,Bournemouth,2019-12-14,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,8.0,4.0,2.0,1.0,18.0,10.0,6.0,2.0
1639,Chelsea,Bournemouth,2022-12-27,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,8.0,5.0,1.0,9.0,14.0,1.0,1.0
2161,Chelsea,Bournemouth,2024-05-19,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,4.0,0.0,2.0,0.0,0.0,4.0,24.0,0.0,7.0
